# Imports??

In [ ]:
import os
import ee
import geemap
from dotenv import load_dotenv

load_dotenv()
project_id = os.getenv("GOOGLE_CLOUD_PROJECT_ID")

ee.Authenticate()
ee.Initialize(project=project_id)

# Helper Functions


In [ ]:
def mask_clouds_landsat(image):
    """Given an Image from the Landsat 8 Collection 2 Tier 1 dataset, applies a mask that removes 
    clouds and shadows."""

    # Select the Quality Assessment band
    qa = image.select('QA_PIXEL')

    # Bits 3 and 4 are Cloud and Cloud Shadow, respectively.
    # We create a mask where these bits are set to 0 (Clear)
    cloud_bit = 1 << 3
    shadow_bit = 1 << 4

    # Combined mask: pixel must have neither cloud nor shadow
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(shadow_bit).eq(0))

    # Apply the mask to the image (making clouds/shadows transparent)
    return image.updateMask(mask)


def add_coords(feature):
    """Given a Feature, adds the longitude and latitude as properties."""
    return feature.set({
        'longitude': feature.geometry().coordinates().get(0),
        'latitude': feature.geometry().coordinates().get(1)
    })

In [ ]:
def get_median(y):
    """Gets the median of the Landsat 8 Collection 2 Tier 1 image bands in a given year"""
    return ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterDate(ee.Date.fromYMD(y, 1, 1), ee.Date.fromYMD(y, 1, 1).advance(1, 'year')) \
        .map(mask_clouds_landsat) \
        .median()

def get_indices(img):
    """Given an Image from the Landsat 8 Collection 2 Tier 1 dataset, creates and adds the 
    Red, Near Infrared, Shortwave Infrared 1 and 2 spectral bands, as well as NDVI, NBR, NDMI indices.
    Learn more about spectral indices here: https://www.geo.university/pages/spectral-indices-in-remote-sensing-and-how-to-interpret-them"""
    # Select original bands
    selected_bands = img.select(['SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'])

    # Calculate indices using normalizedDifference
    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI') # NIR - Red (SR_B5 - SR_B4)
    nbr = img.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')   # NIR - SWIR2 (SR_B5 - SR_B7)
    ndmi = img.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI') # NIR - SWIR1 (SR_B5 - SR_B6)

    # Return original bands plus calculated indices
    return selected_bands.addBands([ndvi, nbr, ndmi])


def process_yearly_landsat(year, base_year):
    """Given a year and base year, gets lagged bands, indices, and year-to-base_year deltas of the Landsat 8 Collection 2 Tier 1 dataset as per get_indices(), from year to base_year"""
    # gets lagged spectral bands, spectral indices, and deltas from year to base_year
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = start_date.advance(1, 'year')


    # Process the target year
    img = get_indices(get_median(year))
    current_img = get_indices(get_median(base_year))
    deltas = img.subtract(current_img).rename(img.bandNames().map(lambda n: ee.String(n).cat('_delta')))
    img = img.addBands(deltas)

    # Process the base year

    # Calculate Lags for labeling - Added .toInt() to avoid '.0' in band names
    lag = ee.Number(base_year).subtract(year)
    lag_suffix = ee.String('_lag').cat(lag.toInt().format())

    # Rename year bands
    img_renamed = img.rename(img.bandNames().map(lambda n: ee.String(n).cat(lag_suffix)))

    return img_renamed \
                .set('year', year) \
                .set('lag', lag) \
                .set('base_year', base_year) \
                .set('system:time_start', start_date.millis())

In [ ]:
def get_data(year:int, location:ee.Geometry, filename:str):
  """Given a year, a location as a Geometry, and a filename, 
     extracts the image bands/indices and targets (whether or not the pixel was deforested in year). Saves to filename."""

  # get targets
  hansen = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
  lossyear = hansen.select('lossyear')
  datamask = hansen.select('datamask')
  treecover = hansen.select('treecover2000')

  land_forest_mask = datamask.eq(1).And(treecover.gt(50))
  target_band = lossyear.unmask(0).updateMask(land_forest_mask).rename('target')
  valid_loss_mask = target_band.eq(0).Or(target_band.eq(year-2000))
  class_band = ee.Image(0).updateMask(valid_loss_mask).where(target_band.eq(year-2000), 1).where(target_band.eq(0), 0).rename('class')

  # sample points to include in the dataset
  sampled_points = class_band.stratifiedSample(
    numPoints=1000,      # Number of points to sample for each class
    classBand='class',    # The name of the band containing the class labels
    region=location,      # The geographical region to sample within
    scale=30,             # The nominal scale (resolution) in meters at which to sample
    geometries=True,      # Include feature geometries (points) in the output
    seed=0                # A random seed for reproducibility
  )
  sampled_points = sampled_points.map(add_coords)

  experimental = sampled_points.filter(ee.Filter.eq('class', 1))
  control = sampled_points.filter(ee.Filter.eq('class', 0))

  full_set = experimental.merge(control)

  # get features
  years = ee.List.sequence(year-4, year-1)
  collection = ee.ImageCollection(years.map(lambda y: process_yearly_landsat(ee.Number(y), year)))

  # merge features and target for our points
  features_image = collection.toBands()
  targets_with_features = features_image.sampleRegions(
      collection=full_set,
      properties=['class', 'longitude', 'latitude'], # Include the 'class', 'longitude', and 'latitude' properties
      scale=30
  )

  # export dataset
  export_task = ee.batch.Export.table.toDrive(
    collection=targets_with_features,
    description=f'exporting {filename}',
    folder='ee_exports',
    fileNamePrefix=filename,
    fileFormat='CSV'
  )

  export_task.start()

In [ ]:
# Get three years of training data in the Amazon Rainforest, Siberian Taiga, and Borneo rainforest. Come final model evaluation, I will also extract 2024 as my test set

amazon_roi = ee.Geometry.Rectangle([-75, -15, -50, 5])
siberian_taiga_roi = ee.Geometry.Rectangle([80, 50, 120, 70])
borneo_roi = ee.Geometry.Rectangle([108, -5, 119, 8])

get_data(2021, amazon_roi, 'amazon_train_2021')
get_data(2022, amazon_roi, 'amazon_train_2022')
get_data(2023, amazon_roi, 'amazon_train_2023')
get_data(2021, siberian_taiga_roi, 'taiga_train_2021')
get_data(2022, siberian_taiga_roi, 'taiga_train_2022')
get_data(2023, siberian_taiga_roi, 'taiga_train_2023')
get_data(2021, borneo_roi, 'borneo_train_2021')
get_data(2022, borneo_roi, 'borneo_train_2022')
get_data(2023, borneo_roi, 'borneo_train_2023')